# 🎬 Live Demo: Portfolio Backtesting System (Standalone)

**DADS 4002 - Database Systems Project**

**✨ Notebook นี้ Standalone 100% - ไม่ต้องพึ่งไฟล์อื่นเลย!**

---

## 🎯 วิธีใช้งาน (ใช้ได้บนเครื่องใหม่ทันที!)

### **เครื่องต้องมี:**
1. ✅ Python 3.8+
2. ✅ MySQL Server ทำงานอยู่
3. ✅ Database `portfolio_backtesting` พร้อมข้อมูล
4. ✅ Libraries: `pip install jupyter mysql-connector-python`

### **ขั้นตอน:**
1. **แก้ไข MySQL Password** ใน Cell 1 (ด้านล่าง)
2. **รัน Cell 1** (Setup) → **Shift+Enter**
3. **รัน Cell 2** (Live Demo) → **Shift+Enter ครั้งเดียว**
4. **เลือก 1-5** ไปเรื่อยๆ จนกว่าจะเลือก 0 เพื่อออก

---

## ⭐ Features

- ✅ **Standalone 100%** - SQL embedded ในไฟล์นี้แล้ว ไม่ต้องอ่านไฟล์อื่น!
- ✅ **ใช้ได้ทันที** - แค่แก้ password แล้ว Run!
- ✅ **Interactive Menu** - เลือก 1-5 ไปเรื่อยๆ

---

---

# 📦 Cell 1: Setup + Install SQL Procedures

## ⚠️ **สำคัญ: แก้ไข MySQL Password**

**หา `MYSQL_CONFIG` ด้านล่างแล้วแก้ไข password**

---

**กด Shift+Enter เพื่อรัน:**

In [ ]:
import mysql.connector
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# ========================================
# ⚠️ แก้ไข PASSWORD ตรงนี้!
# ========================================
MYSQL_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': 'krittanut123456',  # ⬅️ แก้ไขตรงนี้!
    'database': 'portfolio_backtesting'
}
# ========================================

# SQL Statements (Embedded - ไม่ต้องอ่านไฟล์!)
SQL_STATEMENTS = """
-- Drop existing views and procedures
DROP VIEW IF EXISTS vw_weekly_returns;
DROP VIEW IF EXISTS vw_portfolio_summary;
DROP VIEW IF EXISTS vw_etf_performance;

DROP PROCEDURE IF EXISTS sp_calculate_portfolio_return;
DROP PROCEDURE IF EXISTS sp_calculate_sharpe_ratio;
DROP PROCEDURE IF EXISTS sp_calculate_max_drawdown;
DROP PROCEDURE IF EXISTS sp_calculate_volatility;
DROP PROCEDURE IF EXISTS sp_get_etf_correlation;
DROP PROCEDURE IF EXISTS sp_get_top_performers;
DROP PROCEDURE IF EXISTS sp_compare_portfolios;
DROP PROCEDURE IF EXISTS sp_get_portfolio_weights;
"""

print("="*80)
print("🚀 Portfolio Backtesting System - Standalone Setup")
print("="*80)

# Connect to MySQL
print("\n🔌 กำลังเชื่อมต่อ MySQL...")
try:
    conn = mysql.connector.connect(**MYSQL_CONFIG)
    cursor = conn.cursor(dictionary=True)
    print("✅ เชื่อมต่อ MySQL สำเร็จ!")
except mysql.connector.Error as e:
    print(f"❌ Connection Error: {e}")
    print("\n💡 วิธีแก้:")
    print("   1. ตรวจสอบว่า MySQL Server กำลังทำงานอยู่")
    print("   2. แก้ไข password ใน MYSQL_CONFIG ด้านบน")
    raise

print("\n⚙️  กำลังติดตั้ง SQL Stored Procedures & Views...")
print("-"*80)

# Drop existing objects
for stmt in SQL_STATEMENTS.strip().split(';'):
    stmt = stmt.strip()
    if stmt:
        try:
            cursor.execute(stmt)
        except:
            pass

# Create View 1: vw_weekly_returns
cursor.execute("""
CREATE VIEW vw_weekly_returns AS
SELECT
    e.ticker_symbol,
    e.etf_id,
    ph.date,
    ph.adj_close,
    LAG(ph.adj_close) OVER (PARTITION BY e.ticker_symbol ORDER BY ph.date) AS prev_close,
    CASE
        WHEN LAG(ph.adj_close) OVER (PARTITION BY e.ticker_symbol ORDER BY ph.date) IS NOT NULL
        THEN (ph.adj_close - LAG(ph.adj_close) OVER (PARTITION BY e.ticker_symbol ORDER BY ph.date))
             / LAG(ph.adj_close) OVER (PARTITION BY e.ticker_symbol ORDER BY ph.date)
        ELSE NULL
    END AS weekly_return
FROM price_history ph
JOIN etf_master e ON ph.etf_id = e.etf_id
""")
print("  ✅ สร้าง View: vw_weekly_returns")

# Create View 2: vw_portfolio_summary
cursor.execute("""
CREATE VIEW vw_portfolio_summary AS
SELECT
    b.benchmark_id,
    b.benchmark_name,
    b.risk_level,
    COUNT(bh.etf_id) AS num_holdings,
    SUM(bh.target_weight) AS total_weight
FROM benchmark_portfolios b
LEFT JOIN benchmark_holdings bh ON b.benchmark_id = bh.benchmark_id
LEFT JOIN etf_master e ON bh.etf_id = e.etf_id
GROUP BY b.benchmark_id, b.benchmark_name, b.risk_level
""")
print("  ✅ สร้าง View: vw_portfolio_summary")

# Create View 3: vw_etf_performance
cursor.execute("""
CREATE VIEW vw_etf_performance AS
WITH etf_stats AS (
    SELECT
        ticker_symbol,
        etf_id,
        AVG(weekly_return) AS avg_return,
        STDDEV(weekly_return) AS volatility
    FROM vw_weekly_returns
    WHERE weekly_return IS NOT NULL
    GROUP BY ticker_symbol, etf_id
)
SELECT
    e.ticker_symbol,
    e.etf_name,
    ROUND(s.avg_return * 52 * 100, 2) AS annualized_return_pct,
    ROUND(s.volatility * SQRT(52) * 100, 2) AS annualized_volatility_pct,
    ROUND((s.avg_return * 52) / (s.volatility * SQRT(52)), 2) AS sharpe_ratio_approx
FROM etf_stats s
JOIN etf_master e ON s.etf_id = e.etf_id
""")
print("  ✅ สร้าง View: vw_etf_performance")

# Create Stored Procedures
cursor.execute("""
CREATE PROCEDURE sp_calculate_sharpe_ratio(
    IN p_ticker_symbol VARCHAR(10),
    IN p_risk_free_rate DECIMAL(5,4),
    OUT p_sharpe_ratio DECIMAL(8,4)
)
BEGIN
    DECLARE v_avg_return DECIMAL(10,6);
    DECLARE v_stddev DECIMAL(10,6);
    
    SELECT AVG(weekly_return), STDDEV(weekly_return)
    INTO v_avg_return, v_stddev
    FROM vw_weekly_returns
    WHERE ticker_symbol = p_ticker_symbol AND weekly_return IS NOT NULL;
    
    IF v_stddev > 0 THEN
        SET p_sharpe_ratio = ((v_avg_return - (p_risk_free_rate / 52)) / v_stddev) * SQRT(52);
    ELSE
        SET p_sharpe_ratio = 0;
    END IF;
END
""")
print("  ✅ สร้าง Procedure: sp_calculate_sharpe_ratio")

cursor.execute("""
CREATE PROCEDURE sp_get_top_performers(
    IN p_metric VARCHAR(20),
    IN p_top_n INT,
    IN p_start_date DATE,
    IN p_end_date DATE
)
BEGIN
    IF p_metric = 'sharpe' THEN
        SELECT ticker_symbol, etf_name, annualized_return_pct, 
               annualized_volatility_pct, sharpe_ratio_approx AS sharpe_ratio
        FROM vw_etf_performance
        ORDER BY sharpe_ratio_approx DESC
        LIMIT p_top_n;
    END IF;
END
""")
print("  ✅ สร้าง Procedure: sp_get_top_performers")

cursor.execute("""
CREATE PROCEDURE sp_compare_portfolios(
    IN p_start_date DATE,
    IN p_end_date DATE
)
BEGIN
    SELECT
        b.benchmark_name,
        b.risk_level,
        ROUND(AVG(wr.weekly_return) * 52 * 100, 2) AS portfolio_return_pct,
        ROUND(STDDEV(wr.weekly_return) * SQRT(52) * 100, 2) AS annualized_volatility_pct,
        ROUND((AVG(wr.weekly_return) * 52) / (STDDEV(wr.weekly_return) * SQRT(52)), 2) AS sharpe_ratio
    FROM benchmark_portfolios b
    JOIN benchmark_holdings bh ON b.benchmark_id = bh.benchmark_id
    JOIN vw_weekly_returns wr ON bh.etf_id = wr.etf_id
    WHERE wr.date BETWEEN p_start_date AND p_end_date AND wr.weekly_return IS NOT NULL
    GROUP BY b.benchmark_id, b.benchmark_name, b.risk_level
    ORDER BY sharpe_ratio DESC;
END
""")
print("  ✅ สร้าง Procedure: sp_compare_portfolios")

cursor.execute("""
CREATE PROCEDURE sp_get_etf_correlation(
    IN p_ticker1 VARCHAR(10),
    IN p_ticker2 VARCHAR(10),
    OUT p_correlation DECIMAL(8,4)
)
BEGIN
    WITH paired_returns AS (
        SELECT w1.weekly_return AS return1, w2.weekly_return AS return2
        FROM vw_weekly_returns w1
        JOIN vw_weekly_returns w2 ON w1.date = w2.date
        WHERE w1.ticker_symbol = p_ticker1 AND w2.ticker_symbol = p_ticker2
          AND w1.weekly_return IS NOT NULL AND w2.weekly_return IS NOT NULL
    )
    SELECT
        (COUNT(*) * SUM(return1 * return2) - SUM(return1) * SUM(return2)) /
        SQRT((COUNT(*) * SUM(return1 * return1) - SUM(return1) * SUM(return1)) *
             (COUNT(*) * SUM(return2 * return2) - SUM(return2) * SUM(return2)))
    INTO p_correlation
    FROM paired_returns;
END
""")
print("  ✅ สร้าง Procedure: sp_get_etf_correlation")

conn.commit()

# Get statistics
cursor.execute("SELECT COUNT(*) as total FROM etf_master")
etf_count = cursor.fetchone()['total']

cursor.execute("SELECT COUNT(*) as total FROM benchmark_portfolios")
portfolio_count = cursor.fetchone()['total']

cursor.execute("SELECT COUNT(*) as total FROM price_history")
price_count = cursor.fetchone()['total']

print("\n" + "="*80)
print("✅ Setup เสร็จสมบูรณ์!")
print("="*80)
print(f"\n📈 Database Statistics:")
print(f"   - ETFs: {etf_count:,}")
print(f"   - Portfolios: {portfolio_count:,}")
print(f"   - Price History: {price_count:,}")
print("\n🎉 พร้อม Demo! รัน Cell ถัดไป (Shift+Enter)")
print("="*80)

---

# 🎯 Cell 2: Live Demo - Interactive Menu

**กด Shift+Enter ครั้งเดียว แล้วเลือก 1-5 ไปเรื่อยๆ**

---

In [ ]:
def show_menu():
    print("\n" + "="*80)
    print("📈 Data Analytics Menu (SQL-based)")
    print("="*80)
    print("")
    print("1. 🏆 Top Performers - Top 5 ETFs ที่ดีที่สุด")
    print("2. ⚖️  Portfolio Comparison - เปรียบเทียบ Portfolios")
    print("3. 🔗 ETF Correlation - วิเคราะห์ความสัมพันธ์")
    print("4. 📊 Sharpe Ratio Calculator - คำนวณ Sharpe Ratio")
    print("5. 📈 Top 10 Performance - ดูผลตอบแทน")
    print("0. 🚪 Exit - ออกจากโปรแกรม")
    print("")
    print("="*80)

def top_performers(cursor):
    print("\n🏆 Top 5 ETFs (Sharpe Ratio)")
    print("="*80)
    cursor.callproc('sp_get_top_performers', ['sharpe', 5, '2009-01-01', '2025-01-01'])
    for result in cursor.stored_results():
        rows = result.fetchall()
    if rows:
        print(f"\n{'#':<4} {'Ticker':<10} {'ETF Name':<25} {'Return':<10} {'Volatility':<12} {'Sharpe':<8}")
        print("-"*80)
        for i, row in enumerate(rows, 1):
            print(f"{i:<4} {row['ticker_symbol']:<10} {row['etf_name'][:23]:<25} {row['annualized_return_pct']:.2f}%{'':<5} {row['annualized_volatility_pct']:.2f}%{'':<7} {row['sharpe_ratio']:.4f}")
        print("\n💡 Insights: {} มี Sharpe Ratio สูงสุด ({:.4f})".format(rows[0]['ticker_symbol'], rows[0]['sharpe_ratio']))

def portfolio_comparison(cursor):
    print("\n⚖️  Portfolio Comparison")
    print("="*80)
    cursor.callproc('sp_compare_portfolios', ['2020-01-01', '2025-01-01'])
    for result in cursor.stored_results():
        rows = result.fetchall()
    if rows:
        print(f"\n{'Portfolio':<28} {'Risk':<12} {'Return':<10} {'Volatility':<12} {'Sharpe':<8}")
        print("-"*80)
        for row in rows:
            print(f"{row['benchmark_name'][:26]:<28} {row['risk_level'][:10]:<12} {row['portfolio_return_pct']:.2f}%{'':<5} {row['annualized_volatility_pct']:.2f}%{'':<7} {row['sharpe_ratio']:.4f}")
        best = max(rows, key=lambda x: x['sharpe_ratio'])
        print("\n💡 Insights: {} ดีที่สุด (Sharpe: {:.4f})".format(best['benchmark_name'], best['sharpe_ratio']))

def etf_correlation(cursor):
    print("\n🔗 ETF Correlation")
    print("="*80)
    ticker1 = input("\nTicker 1 (เช่น SPY): ").strip().upper()
    ticker2 = input("Ticker 2 (เช่น QQQ): ").strip().upper()
    result_args = cursor.callproc('sp_get_etf_correlation', [ticker1, ticker2, 0])
    corr = result_args[2]
    if corr:
        print(f"\nCorrelation ({ticker1} vs {ticker2}): {corr:.4f}")
        if corr > 0.8:
            print("💡 Insights: ไม่แนะนำถือร่วมกัน (Correlation สูงมาก)")
        elif corr > 0:
            print("💡 Insights: เหมาะสำหรับ Diversification")

def sharpe_ratio_calculator(cursor):
    print("\n📊 Sharpe Ratio Calculator")
    print("="*80)
    ticker = input("\nTicker (เช่น SPY): ").strip().upper()
    try:
        rf = float(input("Risk-Free Rate (เช่น 0.02): "))
    except:
        rf = 0.02
    result_args = cursor.callproc('sp_calculate_sharpe_ratio', [ticker, rf, 0])
    sharpe = result_args[2]
    if sharpe:
        print(f"\nSharpe Ratio ({ticker}): {sharpe:.4f}")
        print(f"💡 Insights: {'Good!' if sharpe > 1 else 'Fair'} ({'ดี' if sharpe > 1 else 'พอใช้'})")

def etf_performance_view(cursor):
    print("\n📈 Top 10 ETFs Performance")
    print("="*80)
    cursor.execute("SELECT ticker_symbol, annualized_return_pct, annualized_volatility_pct, sharpe_ratio_approx FROM vw_etf_performance ORDER BY sharpe_ratio_approx DESC LIMIT 10")
    rows = cursor.fetchall()
    if rows:
        print(f"\n{'#':<4} {'Ticker':<10} {'Return':<12} {'Volatility':<12} {'Sharpe':<10}")
        print("-"*80)
        for i, row in enumerate(rows, 1):
            print(f"{i:<4} {row['ticker_symbol']:<10} {row['annualized_return_pct']:.2f}%{'':<7} {row['annualized_volatility_pct']:.2f}%{'':<7} {row['sharpe_ratio_approx']:.4f}")
        print("\n💡 ใช้ SQL View: vw_etf_performance")

# Main Loop
print("\n🎬 Live Demo Started!")
print("💡 เลือก 1-5 เพื่อ Demo, เลือก 0 เพื่อออก")

while True:
    try:
        show_menu()
        choice = input("\nเลือก (0-5): ").strip()
        
        if choice == '0':
            print("\n👋 Goodbye!")
            break
        elif choice == '1':
            top_performers(cursor)
        elif choice == '2':
            portfolio_comparison(cursor)
        elif choice == '3':
            etf_correlation(cursor)
        elif choice == '4':
            sharpe_ratio_calculator(cursor)
        elif choice == '5':
            etf_performance_view(cursor)
        else:
            print("\n⚠️  เลือก 0-5 เท่านั้น")
        
        input("\n⏎ Enter เพื่อกลับ Menu...")
    except KeyboardInterrupt:
        break
    except Exception as e:
        print(f"\n❌ Error: {e}")

cursor.close()
conn.close()
print("\n✅ Connection closed")

---

# ✅ เสร็จสิ้น

**Notebook นี้:**
- ✅ Standalone 100% - ไม่ต้องพึ่งไฟล์อื่น
- ✅ ใช้ SQL Stored Procedures
- ✅ Interactive Menu
- ✅ เหมาะสำหรับ Demo บนเครื่องใหม่

**Thank you! 🎉**